In [0]:
"""
id: python_3
template: python
templateVersion: 1.0.0
name: customers_data
position:
  x: 844.0960191688738
  y: 454.8666990828854
description:
  text: Load data from input or from a CSV URL if input is empty.
  hash: 60c15041
previewCodeHash: c0a30a6854f5a4ac
previewMode: "1000"
config:
  code: |-
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        # result = spark.createDataFrame([], "col: string")
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"
        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result =spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        # result = spark.createDataFrame([], "col: string")
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/customers.csv"
        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result =spark.createDataFrame(df)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_3.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["python_3.result"])

In [0]:
"""
id: python_3_copy
template: python
templateVersion: 1.0.0
name: shipments_data
position:
  x: 1115.4300114592788
  y: 564.428330070389
description:
  text: Load data from input if available; otherwise read and load a CSV file from a web URL.
  hash: b28e6b0e
previewCodeHash: 422a9c3c2516a0ea
previewMode: "1000"
config:
  code: |-
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        # result = spark.createDataFrame([], "col: string")
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"
        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result =spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        # result = spark.createDataFrame([], "col: string")
        import pandas as pd
        from io import StringIO
        import requests
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"
        response=requests.get(url)
        csv_data=response.content.decode('utf-8')
        df=pd.read_csv(StringIO(csv_data))
        result =spark.createDataFrame(df)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_3_copy.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["python_3_copy.result"])

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 577
  y: 356
description:
  text: Load all columns and rows from the learning_catalog.raw.orders table.
  hash: db02c79b
previewCodeHash: e6b40b6d1c8367a5
previewMode: "1000"
config:
  table_source:
    tableName: learning_catalog.raw.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "learning_catalog.raw.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", True):
    display(ctx["source_0.data"])

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 572.5747522645121
  y: 242.49190499733243
description:
  text: Load all data from the order_items table.
  hash: 383c5cd3
previewCodeHash: 3b62f45ebadd7efc
previewMode: "1000"
config:
  table_source:
    tableName: learning_catalog.raw.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "learning_catalog.raw.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]
if globals().get("ld_display_outputs", True):
    display(ctx["source_1.data"])

In [0]:
"""
id: join_2
template: join
templateVersion: 2.0.0
name: order_and_items
position:
  x: 851
  y: 290
description:
  text: Perform a full join on order_id, exclude right order_id from output.
  hash: c1a0bb0d
previewCodeHash: 85ef5981afe8d169
previewMode: "1000"
config:
  join_type: full
  join_keys:
    - left: order_id
      right: order_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits: []
    ordered: []
  right_columns:
    edits:
      - column: order_id
        checked: false
    ordered: []
input:
  - node: source_1
    input_port: left
    output_port: data
  - node: source_0
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "full",
    "join_keys": [
        {
            "left": "order_id",
            "right": "order_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "order_id",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "left": ctx["source_1.data"],
    "right": ctx["source_0.data"]
}
out = run(config, inputs, spark)
ctx["join_2.joined_data"] = out["joined_data"]
ctx["join_2.left_unmatched"] = out["left_unmatched"]
ctx["join_2.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", True):
    display(ctx["join_2.joined_data"])
    display(ctx["join_2.left_unmatched"])
    display(ctx["join_2.right_unmatched"])

In [0]:
"""
id: join_5
template: join
templateVersion: 2.0.0
name: customer_join
position:
  x: 1112.1809809072774
  y: 395.609218984804
description:
  text: Keep all entries from the left dataset and add matching entries from the right dataset on customer_id, excluding customer_id from the left side.
  hash: "312e5567"
previewCodeHash: 75e4b798f3079041
previewMode: "1000"
config:
  join_type: left
  join_keys:
    - left: customer_id
      right: customer_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits:
      - column: customer_id
        checked: false
    ordered: []
  right_columns:
    edits: []
    ordered: []
input:
  - node: join_2
    input_port: left
    output_port: joined_data
  - node: python_3
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_keys": [
        {
            "left": "customer_id",
            "right": "customer_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [
            {
                "column": "customer_id",
                "checked": False
            }
        ],
        "ordered": []
    },
    "right_columns": {
        "edits": [],
        "ordered": []
    }
}
inputs = {
    "left": ctx["join_2.joined_data"],
    "right": ctx["python_3.result"]
}
out = run(config, inputs, spark)
ctx["join_5.joined_data"] = out["joined_data"]
ctx["join_5.left_unmatched"] = out["left_unmatched"]
ctx["join_5.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", True):
    display(ctx["join_5.joined_data"])
    display(ctx["join_5.left_unmatched"])
    display(ctx["join_5.right_unmatched"])

In [0]:
"""
id: join_6
template: join
templateVersion: 2.0.0
name: ship_order_cus_join
position:
  x: 1529.2602857656884
  y: 476.34809236268677
description:
  text: Keep all records from the left dataset and matching records from the right dataset, excluding the right dataset's order_id column.
  hash: a9db06df
previewCodeHash: 9e6d0a1208c1934c
previewMode: "1000"
config:
  join_type: left
  join_keys:
    - left: order_id
      right: order_id
  join_conditions: ""
  match_case: false
  left_columns:
    edits: []
    ordered: []
  right_columns:
    edits:
      - column: order_id
        checked: false
    ordered: []
input:
  - node: python_3_copy
    input_port: right
    output_port: result
  - node: join_5
    input_port: left
    output_port: joined_data
"""

# generated from the system
from typing import Any, Dict, List
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _qualified_col(side: str, name: str):
    escaped = name.replace("`", "``")
    return F.col("`" + side + "`.`" + escaped + "`")

def _ordered_names(df_cols: List[str], cfg: Dict[str, Any]) -> List[str]:
    ordered: List[str] = cfg.get("ordered") or []
    col_set = set(df_cols)
    placed = set()
    result: List[str] = []
    for name in ordered:
        if name in placed or name not in col_set:
            continue
        placed.add(name)
        result.append(name)
    for name in df_cols:
        if name in placed:
            continue
        placed.add(name)
        result.append(name)
    return result

def _edits_by_column(cfg: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    by_col: Dict[str, Dict[str, Any]] = {}
    for item in cfg.get("edits", []) or []:
        col = item.get("column")
        if col is not None and col not in by_col:
            by_col[col] = item
    return by_col

def _projection(df_left, df_right, left_cfg: Dict[str, Any], right_cfg: Dict[str, Any]):
    left_cols = list(df_left.columns)
    right_cols = list(df_right.columns)
    left_edits = _edits_by_column(left_cfg)
    right_edits = _edits_by_column(right_cfg)
    left_lower = {c.lower() for c in left_cols}

    out = []
    for name in _ordered_names(left_cols, left_cfg):
        edit = left_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("left", name)
        alias = edit.get("alias") if edit else None
        out.append(col.alias(alias) if alias else col.alias(name))
    for name in _ordered_names(right_cols, right_cfg):
        edit = right_edits.get(name)
        if edit is not None and not _is_checked(edit):
            continue
        col = _qualified_col("right", name)
        alias = edit.get("alias") if edit else None
        if alias:
            out.append(col.alias(alias))
        elif name.lower() in left_lower:
            out.append(col.alias("right_" + name))
        else:
            out.append(col.alias(name))
    return out

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_keys: List[Dict[str, str]] = config.get("join_keys", [])
    join_condition = config.get("join_conditions", "")
    join_type = config.get("join_type") or "split_join"
    match_case = config.get("match_case", False)
    left_cfg = config.get("left_columns") or {}
    right_cfg = config.get("right_columns") or {}

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    left_types = {f.name.lower(): f.dataType for f in df_left.schema}
    right_types = {f.name.lower(): f.dataType for f in df_right.schema}

    def key_col(side, name, types):
        col = _qualified_col(side, name)
        if not match_case and isinstance(types.get(name.lower()), StringType):
            return F.upper(F.trim(col))
        return col

    predicates = []
    for key in join_keys:
        predicates.append(
            key_col("left", key["left"], left_types)
            == key_col("right", key["right"], right_types)
        )
    if join_condition:
        predicates.append(F.expr(join_condition))

    join_expr = None
    for predicate in predicates:
        join_expr = predicate if join_expr is None else join_expr & predicate

    is_split = join_type == "split_join"
    matched_how = "inner" if is_split else join_type

    if join_expr is None:
        matched = df_left.join(df_right, how=matched_how)
    else:
        matched = df_left.join(df_right, join_expr, how=matched_how)

    projection = _projection(df_left, df_right, left_cfg, right_cfg)
    if projection:
        matched = matched.select(*projection)

    if is_split:
        if join_expr is None:
            left_unmatched = df_left.join(df_right, how="left_anti")
            right_unmatched = df_right.join(df_left, how="left_anti")
        else:
            left_unmatched = df_left.join(df_right, join_expr, how="left_anti")
            right_unmatched = df_right.join(df_left, join_expr, how="left_anti")
    else:
        left_unmatched = spark.createDataFrame([], df_left.schema)
        right_unmatched = spark.createDataFrame([], df_right.schema)

    return {
        "joined_data": matched,
        "left_unmatched": left_unmatched,
        "right_unmatched": right_unmatched,
    }

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_keys": [
        {
            "left": "order_id",
            "right": "order_id"
        }
    ],
    "join_conditions": "",
    "match_case": False,
    "left_columns": {
        "edits": [],
        "ordered": []
    },
    "right_columns": {
        "edits": [
            {
                "column": "order_id",
                "checked": False
            }
        ],
        "ordered": []
    }
}
inputs = {
    "right": ctx["python_3_copy.result"],
    "left": ctx["join_5.joined_data"]
}
out = run(config, inputs, spark)
ctx["join_6.joined_data"] = out["joined_data"]
ctx["join_6.left_unmatched"] = out["left_unmatched"]
ctx["join_6.right_unmatched"] = out["right_unmatched"]
if globals().get("ld_display_outputs", True):
    display(ctx["join_6.joined_data"])
    display(ctx["join_6.left_unmatched"])
    display(ctx["join_6.right_unmatched"])

In [0]:
"""
id: aggregate_7
template: aggregate
templateVersion: 2.0.0
name: order_by_city
position:
  x: 1789.2602857656884
  y: 476.34809236268677
description:
  text: Group by city and calculate total orders and sum of unit price.
  hash: dd45abb3
previewCodeHash: e6d942905b768279
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: total_order
    - columnExpr:
        expr: round(SUM(unit_price),2)
        type: expr
      fn: "-"
      alias: price_sum
input:
  - node: join_6
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "total_order",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "round(SUM(unit_price),2)",
                "type": "expr"
            },
            "fn": "-",
            "alias": "price_sum",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_6.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_7.aggregated_data"] = out["aggregated_data"]
if globals().get("ld_display_outputs", True):
    display(ctx["aggregate_7.aggregated_data"])

In [0]:
"""
id: sort_8
template: sort
templateVersion: 1.0.0
name: sorted
position:
  x: 2086.1175793022285
  y: 478.9669900780523
description:
  text: Sort data by total_order in descending order.
  hash: 364a6abb
previewCodeHash: 07213c38ebfb0fb4
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: total_order
        type: column
      sortBy: DESC
input:
  - node: aggregate_7
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "total_order",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_7.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_8.sorted_data"] = out["sorted_data"]
if globals().get("ld_display_outputs", True):
    display(ctx["sort_8.sorted_data"])

In [0]:
"""
id: output_9
template: output
templateVersion: 2.0.0
name: learning_catalog.raw.aggregated_orders
position:
  x: 2346.1175793022285
  y: 478.9669900780523
description:
  text: Overwrite table_10 with the input data.
  hash: d0ac1903
previewMode: "1000"
config:
  output_type: table
  catalog: learning_catalog
  schema: raw
  table_name: aggregated_orders
  write_mode: overwrite
input:
  - node: sort_8
    input_port: data
    output_port: sorted_data
"""

# generated from the system
import uuid
from typing import Dict, Any, List

UNIFORM_PROPS = (
    "'delta.enableIcebergCompatV2' = 'true', "
    "'delta.universalFormat.enabledFormats' = 'iceberg', "
    "'delta.columnMapping.mode' = 'name'"
)

def _quote(name: str) -> str:
    if len(name) >= 2 and name.startswith("`") and name.endswith("`"):
        name = name[1:-1].replace("``", "`")
    return "`" + name.replace("`", "``") + "`"

def _qualified(catalog: str, schema: str, table: str) -> str:
    if not table:
        raise ValueError("Output: 'table_name' is required")
    if catalog and not schema:
        raise ValueError("Output: 'schema' is required when 'catalog' is set")
    parts = [_quote(p) for p in (catalog, schema, table) if p]
    return ".".join(parts)

def _tbl_properties_clause(user_props: str, format_value: str) -> str:
    parts: List[str] = []
    if format_value == "uniform":
        parts.append(UNIFORM_PROPS)
    cleaned = (user_props or "").strip().rstrip(",").strip()
    if cleaned:
        parts.append(cleaned)
    if not parts:
        return ""
    return " TBLPROPERTIES (" + ", ".join(parts) + ")"

def _using_clause(format_value: str) -> str:
    if format_value == "iceberg":
        return " USING ICEBERG"
    if format_value in ("delta", "uniform"):
        return " USING DELTA"
    return ""

def _resolve_args(table_properties: str) -> Dict[str, str]:
    import re

    param_names = set(re.findall(r"(?<!:):(\w+)", table_properties or ""))
    if not param_names:
        return {}
    try:
        widgets = dbutils.widgets.getAll()
    except NameError:
        return {}
    return {name: widgets[name] for name in param_names if name in widgets}

def _comment_clause(comment: str) -> str:
    cleaned = (comment or "").strip()
    if not cleaned:
        return ""
    escaped = cleaned.replace("'", "''")
    return f" COMMENT '{escaped}'"

def _cluster_by_clause(mode: str, column: str) -> str:
    if mode == "auto":
        return " CLUSTER BY AUTO"
    if mode == "column" and column:
        return f" CLUSTER BY ({_quote(column)})"
    return ""

SUPPORTED_FILE_TYPES = {"csv", "json"}

MAX_SINGLE_FILE_ROWS = 1_000_000

def _file_path(catalog: str, schema: str, volume: str, file_name: str) -> str:
    if not file_name:
        raise ValueError("Output: 'file_name' is required for a file output")
    if catalog and schema and volume:
        if "/" in file_name or ".." in file_name:
            raise ValueError(
                f"Output: invalid 'file_name' {file_name!r}: it is the final "
                "path segment under the volume and cannot contain '/' or '..'."
            )
        schema_segment = schema.replace(".", "/")
        return f"/Volumes/{catalog}/{schema_segment}/{volume}/{file_name}"
    return file_name

def _with_extension(file_name: str, file_type: str) -> str:
    suffix = f".{file_type}"
    if not file_name or file_name.lower().endswith(suffix):
        return file_name
    return file_name + suffix

def _write_single_file(rows, columns, file_type: str, path: str, append: bool) -> None:
    import csv
    import json
    import os
    import shutil
    import uuid

    def _remove(target: str) -> None:
        if os.path.isdir(target):
            shutil.rmtree(target)
        elif os.path.exists(target):
            os.remove(target)

    def _is_spark_output_dir(target: str) -> bool:
        return os.path.isfile(os.path.join(target, "_SUCCESS"))

    def _stage(write_body) -> None:
        tmp_path = f"{path}.{uuid.uuid4().hex}.tmp"
        try:
            write_body(tmp_path)
        except BaseException:
            _remove(tmp_path)
            raise
        _remove(path)
        os.rename(tmp_path, path)

    if os.path.isdir(path) and not _is_spark_output_dir(path):
        raise ValueError(
            f"Output: cannot write file to {path}: a directory already "
            f"exists at that path."
        )
    appending = append and os.path.isfile(path)
    if file_type == "csv":
        existing_data_rows = 0
        has_existing_header = False
        header = list(columns)
        if appending:
            with open(path, newline="", encoding="utf-8") as handle:
                reader = csv.reader(handle)
                first = next(reader, None)
                if first is not None:
                    header = first
                    has_existing_header = True
                    existing_data_rows = sum(1 for _ in reader)
        if has_existing_header and set(header) != set(columns):
            raise ValueError(
                f"Output: cannot append to {path}: the data's columns "
                f"{sorted(columns)} do not match the existing file's "
                f"columns {sorted(header)}."
            )
        if existing_data_rows + len(rows) > MAX_SINGLE_FILE_ROWS:
            raise ValueError(
                f"Output: appending {len(rows):,} rows would grow {path} past "
                f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
                f"— write to a table instead."
            )

        def _write_csv(tmp_path: str) -> None:
            with open(tmp_path, "w", newline="", encoding="utf-8") as out:
                if has_existing_header:
                    with open(path, newline="", encoding="utf-8") as src:
                        shutil.copyfileobj(src, out)
                else:
                    csv.writer(out).writerow(header)
                writer = csv.writer(out)
                for row in rows:
                    writer.writerow([row[c] for c in header])

        _stage(_write_csv)
        return

    records = []
    if appending:
        with open(path, encoding="utf-8") as handle:
            existing = json.load(handle)
        records = existing if isinstance(existing, list) else [existing]
    if len(records) + len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: appending {len(rows):,} rows would grow {path} past "
            f"the single-file export limit ({MAX_SINGLE_FILE_ROWS:,} rows) "
            f"— write to a table instead."
        )
    records.extend({c: row[c] for c in columns} for row in rows)

    def _write_json(tmp_path: str) -> None:
        with open(tmp_path, "w", encoding="utf-8") as handle:
            json.dump(records, handle, default=str, indent=2)

    _stage(_write_json)

def _run_file(config: Dict[str, Any], df, spark) -> None:
    file_type = config.get("file_type", "csv")
    if file_type not in SUPPORTED_FILE_TYPES:
        raise ValueError(
            f"Output: file_type '{file_type}' is not yet supported. Use csv or json."
        )
    path = _file_path(
        config.get("catalog", ""),
        config.get("schema", ""),
        config.get("volume", ""),
        _with_extension(config.get("file_name", ""), file_type),
    )
    append = config.get("write_mode", "overwrite") == "append"
    rows = df.limit(MAX_SINGLE_FILE_ROWS + 1).collect()
    if len(rows) > MAX_SINGLE_FILE_ROWS:
        raise ValueError(
            f"Output: result too large for single-file export "
            f"(over {MAX_SINGLE_FILE_ROWS:,} rows) — write to a table instead."
        )
    _write_single_file(rows, df.columns, file_type, path, append)

def _run_materialized_view(
    config: Dict[str, Any], source_view: str, spark
) -> None:
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    view_name = config.get("view_name", "")
    if not view_name:
        raise ValueError("Output: 'view_name' is required for a materialized view")
    full_name = _qualified(catalog, schema, view_name)
    cluster_by = _cluster_by_clause(
        config.get("cluster_by_mode", "auto"), config.get("cluster_by_column", "") or ""
    )
    comment = _comment_clause(config.get("comment", "") or "")
    stmt = (
        f"CREATE OR REFRESH MATERIALIZED VIEW {full_name}{cluster_by}{comment} "
        f"AS SELECT * FROM {_quote(source_view)}"
    )
    spark.sql(stmt)

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    output_type = config.get("output_type", "table")
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")
    write_mode = config.get("write_mode", "overwrite")
    merge_keys: List[str] = [k for k in (config.get("merge_keys") or []) if k]
    format_value = config.get("format", "default")
    table_properties = config.get("table_properties", "") or ""

    if output_type == "file":
        _run_file(config, df, spark)
        return {}

    source_view = f"_lb_output_v2_src_{uuid.uuid4().hex}"
    df.createOrReplaceTempView(source_view)

    if output_type == "materialized_view":
        try:
            _run_materialized_view(config, source_view, spark)
        finally:
            spark.catalog.dropTempView(source_view)
        return {}

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    full_name = _qualified(catalog, schema, table_name)

    if write_mode == "merge" and not merge_keys:
        raise ValueError("Output: 'merge_keys' is required when write_mode is merge")

    try:
        args = _resolve_args(table_properties)

        using = _using_clause(format_value)
        tblprops = _tbl_properties_clause(table_properties, format_value)

        if write_mode == "append":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            insert_stmt = (
                f"INSERT INTO {full_name} BY NAME "
                f"SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(insert_stmt, args=args) if args else spark.sql(insert_stmt)
        elif write_mode == "merge":
            create_stmt = (
                f"CREATE TABLE IF NOT EXISTS {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)} WHERE 1=0"
            )
            spark.sql(create_stmt, args=args) if args else spark.sql(create_stmt)
            on_clause = " AND ".join(
                f"t.{_quote(k)} = s.{_quote(k)}" for k in merge_keys
            )
            merge_stmt = (
                f"MERGE INTO {full_name} t "
                f"USING {_quote(source_view)} s "
                f"ON {on_clause} "
                f"WHEN MATCHED THEN UPDATE SET * "
                f"WHEN NOT MATCHED THEN INSERT *"
            )
            spark.sql(merge_stmt, args=args) if args else spark.sql(merge_stmt)
        else:
            stmt = (
                f"CREATE OR REPLACE TABLE {full_name}{using}{tblprops} "
                f"AS SELECT * FROM {_quote(source_view)}"
            )
            spark.sql(stmt, args=args) if args else spark.sql(stmt)
    except Exception as exc:
        if table_properties.strip():
            raise ValueError(
                "Output: failed to write the table. Check the 'table_properties' "
                "and 'schema' fields for invalid SQL. Underlying error: "
                f"{exc}"
            ) from exc
        raise
    finally:
        spark.catalog.dropTempView(source_view)
    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "output_type": "table",
    "catalog": "learning_catalog",
    "schema": "raw",
    "table_name": "aggregated_orders",
    "write_mode": "overwrite"
}
inputs = {
    "data": ctx["sort_8.sorted_data"]
}
out = run(config, inputs, spark)